# 🗑️ Klasifikasi Sampah dengan SqueezeNet 1.1 (PyTorch) + KaggleHub

Notebook ini mengadaptasi kode `train_245_class.py` (PyTorch) dengan mengganti backbone dari **MobileNetV2** menjadi **SqueezeNet 1.1**, dan dataset diunduh otomatis via **KaggleHub**.

**Dataset:** [Recyclable and Household Waste Classification](https://www.kaggle.com/datasets/alistairking/recyclable-and-household-waste-classification) — 15.000 gambar, 30 kategori, 256×256 px.

**Model:** SqueezeNet 1.1 (pretrained ImageNet) — ringan, ~1.2M parameter, cocok untuk CPU.

## 1. Setup & Instalasi

In [ ]:
%pip install kagglehub torch torchvision pillow tqdm scikit-learn matplotlib seaborn

: 

In [ ]:
import os
import shutil
import random
import copy
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('Device:', device)
print('PyTorch:', torch.__version__)

## 2. Download Dataset via KaggleHub

Dataset ini punya struktur **bertingkat**: `images/<kategori_item>/default/*.png` dan `images/<kategori_item>/real_world/*.png`.

Karena `torchvision.datasets.ImageFolder` hanya mendukung struktur datar (`folder_kelas/gambar`), kita perlu **flatten** struktur tersebut ke format sementara: `flat/<nama_kelas>/gambar.png`. Nama kelas = nama subfolder (`Plastic water bottles`, `Glass beverage bottles`, dst).

Kita juga akan **split** dataset menjadi `train/`, `val/`, `test/` (karena dataset asli tidak menyediakan split).

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")
print('Dataset path:', dataset_path)

# Cari folder 'images'
IMAGES_ROOT = None
for root, dirs, files in os.walk(dataset_path):
    if os.path.basename(root) == 'images':
        IMAGES_ROOT = root
        break
print('IMAGES_ROOT:', IMAGES_ROOT)

# Lihat struktur
categories = sorted(os.listdir(IMAGES_ROOT))
print(f'Jumlah kategori item: {len(categories)}')
for c in categories[:10]:
    print(' -', c)

## 3. Flatten + Split Dataset (adaptasi `data_set_split()` dari `utils.py`)

Fungsi `data_set_split()` di proyek asli butuh struktur datar. Kita buat dulu struktur datar sementara `flat_all/<nama_kelas>/<file>.png`, lalu split ke `train/val/test` dengan rasio **80/10/10** — sama seperti kode asli.

In [ ]:
WORK_DIR = Path('./waste_squeezenet_data')
FLAT_DIR = WORK_DIR / 'flat_all'
SPLIT_DIR = WORK_DIR / 'split'
FLAT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# 3a. Flatten: gabungkan 'default' + 'real_world' ke satu folder per kelas
print('Flattening...')
for cat in tqdm(categories):
    cat_dir = Path(IMAGES_ROOT) / cat
    if not cat_dir.is_dir():
        continue
    out_dir = FLAT_DIR / cat
    out_dir.mkdir(parents=True, exist_ok=True)
    for split_name in ['default', 'real_world']:
        sub = cat_dir / split_name
        if not sub.is_dir():
            continue
        for img_file in sub.iterdir():
            if img_file.suffix.lower() in ('.png', '.jpg', '.jpeg'):
                # Prefix nama file agar tidak bentrok antara default & real_world
                dst = out_dir / f'{split_name}_{img_file.name}'
                if not dst.exists():
                    shutil.copy2(img_file, dst)

print('Selesai flatten.')
print('Total kelas:', len(os.listdir(FLAT_DIR)))

In [ ]:
# 3b. Split train/val/test = 80/10/10 (adaptasi data_set_split dari utils.py)
def data_set_split(src_data_folder, target_data_folder, train_scale=0.8, val_scale=0.1, test_scale=0.1):
    print('Mulai split dataset...')
    class_names = sorted(os.listdir(src_data_folder))
    split_names = ['train', 'val', 'test']
    for split_name in split_names:
        split_path = os.path.join(target_data_folder, split_name)
        os.makedirs(split_path, exist_ok=True)
        for class_name in class_names:
            os.makedirs(os.path.join(split_path, class_name), exist_ok=True)

    for class_name in class_names:
        current_class_data_path = os.path.join(src_data_folder, class_name)
        current_all_data = os.listdir(current_class_data_path)
        random.shuffle(current_all_data)
        n = len(current_all_data)
        train_stop = int(n * train_scale)
        val_stop = int(n * (train_scale + val_scale))
        train_folder = os.path.join(target_data_folder, 'train', class_name)
        val_folder   = os.path.join(target_data_folder, 'val', class_name)
        test_folder  = os.path.join(target_data_folder, 'test', class_name)
        for i, img_name in enumerate(current_all_data):
            src = os.path.join(current_class_data_path, img_name)
            if i < train_stop:
                dst = train_folder
            elif i < val_stop:
                dst = val_folder
            else:
                dst = test_folder
            target_path = os.path.join(dst, img_name)
            if not os.path.exists(target_path):
                shutil.copy2(src, target_path)
        print(f'{class_name:40s} -> train={train_stop}, val={val_stop-train_stop}, test={n-val_stop}')

if not (SPLIT_DIR / 'train').exists() or len(os.listdir(SPLIT_DIR / 'train')) == 0:
    data_set_split(str(FLAT_DIR), str(SPLIT_DIR))
else:
    print('Split sudah ada, skip.')

print('Selesai split.')

In [ ]:
for phase in ['train', 'val', 'test']:
    n = sum(len(fs) for _, _, fs in os.walk(SPLIT_DIR / phase))
    print(f'{phase:5s}: {n} images')

## 4. DataLoader (adaptasi `data_load()` dari `train_245_class.py`)

Transformasi sama seperti kode asli (RandomResizedCrop + HorizontalFlip + Normalize ImageNet).

In [ ]:
def data_load(data_dir=str(SPLIT_DIR), batch_size=32):
    data_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        'val': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }
    data_transforms['test'] = data_transforms['val']

    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                      for x in ['train', 'val', 'test']}
    dataloaders = {x: DataLoader(image_datasets[x], batch_size=batch_size,
                                  shuffle=(x=='train'), num_workers=2, pin_memory=True)
                   for x in ['train', 'val', 'test']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
    class_names = image_datasets['train'].classes
    return dataloaders, dataset_sizes, class_names

dataloaders, dataset_sizes, class_names = data_load(batch_size=32)
print('Jumlah kelas:', len(class_names))
print('Train size :', dataset_sizes['train'])
print('Val size   :', dataset_sizes['val'])
print('Test size  :', dataset_sizes['test'])

## 5. Bangun Model SqueezeNet 1.1 (adaptasi `train_main()` dari `train_245_class.py`)

Kita pakai `models.squeezenet1_1(weights='IMAGENET1K_V1')`. Classifier SqueezeNet berbentuk `Sequential(Dropout, Conv2d, ReLU, AdaptiveAvgPool2d)`. Untuk mengganti jumlah output, kita ganti `classifier[1]` (Conv2d) menjadi `nn.Conv2d(512, num_classes, kernel_size=1)`.

In [ ]:
def build_squeezenet(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = models.SqueezeNet1_1_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.squeezenet1_1(weights=weights)
    # Ganti final conv di classifier (index 1) agar output = num_classes
    in_channels = model.classifier[1].in_channels
    model.classifier[1] = nn.Conv2d(in_channels, num_classes, kernel_size=1)
    model.num_classes = num_classes
    # Re-init final conv (mengikuti pola kode asli SqueezeNet)
    nn.init.normal_(model.classifier[1].weight, mean=0.0, std=0.01)
    nn.init.constant_(model.classifier[1].bias, 0)
    return model

model = build_squeezenet(num_classes=len(class_names), pretrained=True).to(device)
print(model.classifier)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

## 6. Training Loop (adaptasi `train_model()` dari `train_245_class.py`)

In [ ]:
def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs=20):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 20)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            with tqdm(total=dataset_sizes[phase], desc=phase) as pbar:
                for inputs, labels in dataloaders[phase]:
                    inputs = inputs.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                    pbar.update(inputs.size(0))

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = (running_corrects.double() / dataset_sizes[phase]).item()
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
NUM_EPOCHS = 20
MODEL_PATH = 'squeezenet1_1_waste.pth'

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

model, history = train_model(
    model, dataloaders, dataset_sizes,
    criterion, optimizer, scheduler,
    num_epochs=NUM_EPOCHS
)

torch.save(model, MODEL_PATH)
print('Model disimpan ke', MODEL_PATH)

## 7. Visualisasi Kurva Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_acc'], label='Train')
axes[0].plot(history['val_acc'],   label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['train_loss'], label='Train')
axes[1].plot(history['val_loss'],   label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.savefig('results_squeezenet.png', dpi=100)
plt.show()

## 8. Evaluasi pada Test Set + Confusion Matrix (adaptasi `zhibiao()` dari `test_model.py`)

In [ ]:
def evaluate(model, loader, class_names, device):
    model.eval()
    real, pred = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='eval'):
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            _, p = torch.max(outputs, 1)
            pred.extend(p.cpu().numpy().tolist())
            real.extend(labels.numpy().tolist())
    return real, pred

test_loader = dataloaders['test']
y_true, y_pred = evaluate(model, test_loader, class_names, device)

acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f'Test accuracy: {acc*100:.2f}%')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix — SqueezeNet 1.1 (Test Acc: {acc*100:.2f}%)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

## 9. Prediksi Single Image (adaptasi `single_img_predict()` dari `test4singleimg.py`)

In [ ]:
def predict_single(img_path, model, class_names, device=device):
    model.eval()
    tfm = transforms.Compose([
        transforms.Resize([224, 224]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(img_path).convert('RGB')
    x = tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
        probs = torch.softmax(out, dim=1)
        conf, idx = torch.max(probs, 1)
    return class_names[idx.item()], conf.item()

# Uji dengan sampel dari test set
test_ds = datasets.ImageFolder(str(SPLIT_DIR / 'test'))
sample_idx = random.randint(0, len(test_ds) - 1)
sample_path, true_idx = test_ds.samples[sample_idx]
pred_class, conf = predict_single(sample_path, model, class_names)
true_class = class_names[true_idx]

print(f'File       : {sample_path}')
print(f'True       : {true_class}')
print(f'Predicted  : {pred_class}  (conf={conf:.3f})')

plt.figure(figsize=(4, 4))
plt.imshow(Image.open(sample_path))
plt.title(f'Pred: {pred_class}\nConf: {conf:.2f}')
plt.axis('off')
plt.show()

## 10. (Opsional) Upload Gambar Sendiri

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# for fname in uploaded:
#     pred_class, conf = predict_single(fname, model, class_names)
#     print(f'{fname} -> {pred_class} ({conf:.3f})')
#     plt.imshow(Image.open(fname)); plt.title(pred_class); plt.axis('off'); plt.show()

## 11. Simpan & Unduh Model

In [ ]:
print('Model:', MODEL_PATH, '—', os.path.getsize(MODEL_PATH) / 1e6, 'MB')
# Di Colab:
# from google.colab import files
# files.download(MODEL_PATH)